# Parsing quality, failure analysis, and how fields are encoded

How each parsed field is encoded end to end, how malformed/`"not mentioned"` outputs are handled, parse quality vs ScanRefer's `object_name`, and a manual assessment on a stratified subset.

Reads `data_parsing/final_parsing/`, `data_parsing/final_parsing_tokenized/`, ScanRefer's JSON, and — where present — `predictions.p`.

In [ ]:
import os, sys, json, collections, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO = "/content/drive/MyDrive/3D-VG" if IN_COLAB else os.path.abspath(".")
if not os.path.isdir(os.path.join(REPO, "experiments")):
    REPO = os.path.abspath("..")
if IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
os.chdir(REPO); sys.path.insert(0, REPO)

SPLIT       = "val"
PARSE_RAW   = "data_parsing/final_parsing"
PARSE_TOK   = "data_parsing/final_parsing_tokenized"
PREDICTIONS = "outputs/2024-12-18_20-40-38_3DVG-FIXED/predictions.p"
OUT_DIR     = "outputs/analysis/parse_quality"
SAMPLE_N, WRONG_FRACTION, SEED = 200, 0.4, 42
os.makedirs(OUT_DIR, exist_ok=True)
print("repo:", REPO, "| split:", SPLIT)

In [ ]:
from experiments.ablation.parsing import (
    FIELD_MAX_TOKENS, NOT_MENTIONED_TOKENS, match_kinds, tokenize, tokenize_field)

raw = json.load(open(f"{PARSE_RAW}/parsed_result_{SPLIT}.json"))
tok = json.load(open(f"{PARSE_TOK}/tokenized_parsed_result_{SPLIT}.json"))
scanrefer = json.load(open(f"data/ScanRefer_filtered_{SPLIT}.json"))
FIELDS = tuple(FIELD_MAX_TOKENS)

records = []
for r in scanrefer:
    p = raw.get(r["scene_id"], {}).get(r["object_id"], {}).get(r["ann_id"])
    t = tok.get(r["scene_id"], {}).get(r["object_id"], {}).get(r["ann_id"])
    if p is None or t is None: continue
    records.append(dict(scene_id=r["scene_id"], object_id=r["object_id"], ann_id=r["ann_id"],
                        object_name=r["object_name"], description=r["description"],
                        raw=p, tok=t))
print(f"{len(scanrefer)} {SPLIT} annotations, {len(records)} joined to the parse cache "
      f"({len(scanrefer)-len(records)} missing)")
print("caps:", FIELD_MAX_TOKENS, " | absent field ->", NOT_MENTIONED_TOKENS)

## 1 · The encoding chain

One annotation, followed from the LLM's JSON to the tensor the GRU sees.

In [ ]:
from IPython.display import Markdown, display

def table(headers, rows):
    w = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
    out = ["| " + " | ".join(str(h).ljust(w[i]) for i, h in enumerate(headers)) + " |",
           "|" + "|".join("-" * (x + 2) for x in w) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |")
    return "\n".join(out)

ex = next(r for r in records if r["raw"]["adjectives"].strip().lower() == "not mentioned")
print("description:", ex["description"][:110], "\n")
rows = []
for f in FIELDS:
    rows.append([f, repr(ex["raw"][f])[:46], str(ex["tok"][f])[:52],
                 len(ex["tok"][f]), FIELD_MAX_TOKENS[f]])
display(Markdown("**Stage 1-2 — LLM phrase → token list**\n\n" +
                 table(["field", "raw (parsed_result)", "tokenized", "len", "cap"], rows) +
                 "\n\nStage 3, in `lib/dataset.py`: `_transform_parsed` allocates a "
                 "`(cap, 300)` float array and fills row *i* with `glove[token[i]]`, or "
                 "`glove['unk']` when the token is out of vocabulary. Rows beyond the "
                 "field length stay **zero**.\n\n"
                 "Stage 4: `tgt_len` / `adj_len` / `ngh_len` are `len(token_list)` — taken "
                 "**unclipped** — and handed to `pack_padded_sequence`."))

### Two invariants this encoding depends on

`pack_padded_sequence` raises on length 0 or a length exceeding the sequence dimension — checked below over the whole split.

In [ ]:
viol = collections.Counter(); maxlen = collections.Counter(); atcap = collections.Counter()
badtype = collections.Counter()
for r in records:
    for f, cap in FIELD_MAX_TOKENS.items():
        L = len(r["tok"][f])
        maxlen[f] = max(maxlen[f], L)
        if L == 0:   viol[(f, "empty")] += 1
        if L > cap:  viol[(f, "over cap")] += 1
        if L == cap: atcap[f] += 1
        if not isinstance(r["raw"][f], str): badtype[(f, type(r["raw"][f]).__name__)] += 1

rows = [[f, FIELD_MAX_TOKENS[f], maxlen[f], atcap[f],
         viol[(f, "empty")], viol[(f, "over cap")]] for f in FIELDS]
display(Markdown("**Schema check over the whole split**\n\n" +
                 table(["field", "cap", "longest seen", "exactly at cap",
                        "empty (fatal)", "over cap (fatal)"], rows)))
print("non-string raw values:", dict(badtype) or "none — every field is a string")
print("\nVERDICT:", "PASS — no empty field, no field over cap, no malformed type"
      if not viol and not badtype else f"FAIL — {dict(viol)} {dict(badtype)}")

## 2 · `"not mentioned"` and malformed output

The literal string the pipeline writes for an absent field — never `null`, `""` or `[]`, which would be fatal downstream. Tokenises to two ordinary GloVe words.

In [ ]:
nm = collections.Counter(); n = len(records)
for r in records:
    for f in FIELDS:
        if r["raw"][f].strip().lower() == "not mentioned": nm[f] += 1
rows = [[f, nm[f], f"{100*nm[f]/n:.2f}%", str(tokenize_field("not mentioned", f))] for f in FIELDS]
display(Markdown(f"**How often each field is declined** (n={n})\n\n" +
                 table(["field", "'not mentioned'", "rate", "tokenises to"], rows)))

In [ ]:
mismatch = collections.Counter(); examples = []
for r in records:
    for f in FIELDS:
        mine = tokenize_field(r["raw"][f], f)
        if mine != r["tok"][f]:
            mismatch[f] += 1
            if len(examples) < 5: examples.append((f, r["raw"][f], mine, r["tok"][f]))

total_mm = sum(mismatch.values())
print(f"fields where re-tokenising the raw phrase differs from the cache: "
      f"{total_mm} of {n*3} ({100*total_mm/(n*3):.2f}%)  {dict(mismatch)}\n")
for f, rawv, mine, cached in examples[:3]:
    print(f"  {f}: {rawv!r}")
    print(f"    this repo's tokenizer : {mine}")
    print(f"    shipped cache         : {cached}\n")

**The difference.** The shipped cache keeps hyphenated compounds whole (`left-to-right`); this repo's tokenizer splits them, since the GPT pipeline that produced the cache isn't in this repository and used a different tokenizer.

This matters: a hyphenated token not in GloVe collapses to `<unk>`, whereas the split form would contribute real word vectors.

In [ ]:
oov_rate = None          # set below when GloVe is available
hyphen_tokens = collections.Counter(); hyphen_by_field = collections.Counter()
for r in records:
    for f in FIELDS:
        for t in r["tok"][f]:
            if "-" in t and len(t) > 1:
                hyphen_tokens[t] += 1; hyphen_by_field[f] += 1
print(f"hyphenated tokens in the {SPLIT} cache: {sum(hyphen_by_field.values())} occurrences, "
      f"{len(hyphen_tokens)} unique  {dict(hyphen_by_field)}")
print("most common:", ", ".join(f"{t}({c})" for t, c in hyphen_tokens.most_common(10)))

GLOVE = "data/glove.p"
oov_rate = None
if os.path.isfile(GLOVE):
    import pickle
    print("\nloading GloVe (~30 s, 933 MB)...")
    glove = pickle.load(open(GLOVE, "rb"))
    inv = {t for t in hyphen_tokens if t in glove}
    oov_h = set(hyphen_tokens) - inv
    tot = oov = 0
    for r in records:
        for f in FIELDS:
            for t in r["tok"][f]:
                tot += 1
                if t not in glove: oov += 1
    oov_rate = 100 * oov / tot
    print(f"  hyphenated tokens in GloVe : {len(inv)}/{len(hyphen_tokens)} "
          f"({100*len(inv)/len(hyphen_tokens):.0f}%)")
    print(f"  hyphenated and OOV -> <unk>: {sorted(oov_h)[:10]}")
    print(f"\n  whole cache: {tot} tokens, {oov} OOV -> <unk> ({oov_rate:.2f}%)")
else:
    print("\n(GloVe not present; skipping the vocabulary check)")

## 3 · Parse quality against ground truth

ScanRefer's `object_name` is free ground truth for the target slot. Three criteria of increasing leniency, from `experiments/ablation/parsing.py`:

- `exact` — normalised strings identical
- `substring` — either contains the other
- `fuzzy` — token overlap, or edit distance ≤ 1 on the head word

In [ ]:
kinds = collections.Counter()
for r in records:
    k = match_kinds(r["raw"]["target"], r["object_name"])
    r["match"] = k
    for name, hit in k.items():
        if hit: kinds[name] += 1
rows = [[c, kinds[c], f"{100*kinds[c]/len(records):.2f}%"] for c in ("exact","substring","fuzzy")]
display(Markdown(f"**Target-field accuracy, all {len(records)} {SPLIT} annotations**\n\n" +
                 table(["criterion", "correct", "accuracy"], rows)))

### A failure taxonomy for the misses

A miss against `object_name` isn't automatically a parser error: the writer may have used a different word than the annotator (vocabulary gap, not a parse error), or the parser may have produced something the description never said (genuine error).

In [ ]:
bucket = collections.Counter(); by_bucket = collections.defaultdict(list)
for r in records:
    desc = r["description"].lower()
    words = [w for w in r["raw"]["target"].lower().split() if len(w) > 2]
    in_desc = bool(words) and all(w in desc for w in words)
    declined = r["raw"]["target"].strip().lower() == "not mentioned"
    if r["match"]["fuzzy"]:   b = "correct"
    elif declined:            b = "declined (target 'not mentioned')"
    elif in_desc:             b = "vocabulary gap (parser copied the description)"
    else:                     b = "parser error (word not in the description)"
    r["bucket"] = b; bucket[b] += 1
    if len(by_bucket[b]) < 8: by_bucket[b].append(r)

order = ["correct", "vocabulary gap (parser copied the description)",
         "declined (target 'not mentioned')", "parser error (word not in the description)"]
rows = [[b, bucket[b], f"{100*bucket[b]/len(records):.2f}%"] for b in order if bucket[b]]
display(Markdown("**Failure taxonomy of the target field**\n\n" +
                 table(["outcome", "n", "share"], rows) +
                 f"\n\nGenuine parser errors: **{100*bucket[order[3]]/len(records):.2f}%** "
                 f"of the split."))

In [ ]:
for b in order[1:]:
    if not by_bucket[b]: continue
    print(f"\n=== {b} ===")
    for r in by_bucket[b][:5]:
        print(f"  annotated={r['object_name']:<20} parsed={r['raw']['target'][:24]:<26}")
        print(f"    {r['description'][:98]}")

### Which object classes the parser struggles with

In [ ]:
cls_tot = collections.Counter(); cls_miss = collections.Counter()
for r in records:
    cls_tot[r["object_name"]] += 1
    if not r["match"]["fuzzy"]: cls_miss[r["object_name"]] += 1
worst = sorted(((c, cls_miss[c], cls_tot[c], 100*cls_miss[c]/cls_tot[c])
                for c in cls_tot if cls_tot[c] >= 40), key=lambda x: -x[3])[:12]
display(Markdown("**Highest target-miss rate, classes with n ≥ 40**\n\n" +
                 table(["class", "missed", "n", "miss rate"],
                       [[c, m, t, f"{p:.1f}%"] for c, m, t, p in worst])))

## 4 · Figures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .3})
OK, WARN, BAD, INK = "#009E73", "#E69F00", "#D55E00", "#222222"

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.6),
                             gridspec_kw={"width_ratios": [1, 1.25]})

# accuracy by criterion
crit = ["exact", "substring", "fuzzy"]
vals = [100*kinds[c]/len(records) for c in crit]
bars = a1.bar(crit, vals, color=[BAD, WARN, OK], edgecolor="white", width=.6)
for b, v in zip(bars, vals):
    a1.text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha="center", fontweight="bold")
a1.set_ylabel("target-field accuracy (%)"); a1.set_ylim(0, 105)
a1.set_title(f"Target vs object_name  (n={len(records):,})")
a1.grid(axis="x", visible=False)

# taxonomy
lab = ["correct", "vocabulary\ngap", "declined", "parser\nerror"]
share = [100*bucket[b]/len(records) for b in order]
cols = [OK, WARN, "#BBBBBB", BAD]
left = 0
for l, s, c in zip(lab, share, cols):
    a2.barh([0], [s], left=left, color=c, edgecolor="white",
            label=f"{l.replace(chr(10), ' ')}  {s:.2f}%")
    if s > 25:                      # only the dominant segment gets an inside label
        a2.text(left+s/2, 0, f"{l}\n{s:.1f}%", ha="center", va="center",
                fontsize=9, color="white", fontweight="bold")
    left += s
a2.set_yticks([]); a2.set_xlabel("% of annotations"); a2.set_xlim(0, 100)
a2.set_title("What the 5.9% of misses actually are")
a2.grid(axis="y", visible=False)
# the three failure segments are too thin to label in place -- annotate them instead
tail = 100 - share[0]
a2.annotate(f"the {tail:.1f}% that miss:\n"
            + "\n".join(f"{l.replace(chr(10),' ')} {s:.2f}%"
                        for l, s in zip(lab[1:], share[1:])),
            xy=(100 - tail/2, .38), xytext=(66, .95), fontsize=8, ha="center",
            arrowprops=dict(arrowstyle="-", color=INK, lw=.8))
a2.set_ylim(-.5, 1.35)
a2.legend(loc="upper center", bbox_to_anchor=(.5, -.30), ncol=2, fontsize=7.5,
          frameon=False)
fig.tight_layout(); plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.4),
                             gridspec_kw={"width_ratios": [1, 1.3]})

# how often each field is declined
x = np.arange(len(FIELDS)); rate = [100*nm[f]/len(records) for f in FIELDS]
b = a1.bar(x, rate, color=["#0072B2", "#56B4E9", "#9AD0F5"], edgecolor="white", width=.6)
for bb, v, f in zip(b, rate, FIELDS):
    a1.text(bb.get_x()+bb.get_width()/2, v+.4, f"{v:.2f}%", ha="center", fontsize=8.5,
            fontweight="bold")
a1.set_xticks(x); a1.set_xticklabels(FIELDS); a1.set_ylabel("'not mentioned' (%)")
a1.set_title("How often a field is declined"); a1.set_ylim(0, max(rate)*1.3)
a1.grid(axis="x", visible=False)

# worst classes
cs = worst[:9][::-1]
y = np.arange(len(cs))
a2.barh(y, [p for *_ , p in cs], color=BAD, edgecolor="white")
for i, (c, m, t, p) in enumerate(cs):
    a2.text(p+.4, i, f"{p:.0f}%  ({m}/{t})", va="center", fontsize=8)
a2.set_yticks(y); a2.set_yticklabels([c for c, *_ in cs], fontsize=8.5)
a2.set_xlabel("target miss rate (%)"); a2.set_xlim(0, max(p for *_, p in cs)*1.35)
a2.set_title("Classes the parser most often misses  (n ≥ 40)")
a2.grid(axis="y", visible=False)
fig.tight_layout(); plt.show()

## 5 · Does a declined field cost grounding accuracy?

Needs `predictions.p`; skipped if absent. Read as association, not cause — short descriptions supply fewer fields and are independently harder.

In [ ]:
import pickle
if os.path.isfile(PREDICTIONS):
    P = pickle.load(open(PREDICTIONS, "rb"))
    rows, groups = [], collections.defaultdict(list)
    for r in records:
        try: v = P[r["scene_id"]][r["object_id"]][r["ann_id"]]["iou"]
        except Exception: continue
        for f in ("adjectives", "neighbors"):
            declined = r["raw"][f].strip().lower() == "not mentioned"
            groups[(f, declined)].append(v >= 0.25)
    for f in ("adjectives", "neighbors"):
        a, b = groups[(f, False)], groups[(f, True)]
        if not a or not b: continue
        fa, fb = 100*sum(a)/len(a), 100*sum(b)/len(b)
        rows.append([f, f"{len(a):,}", f"{fa:.2f}%", f"{len(b):,}", f"{fb:.2f}%",
                     f"{fb-fa:+.2f} pp"])
    display(Markdown("**Acc@0.25 by whether the field was supplied**\n\n" +
                     table(["field", "n present", "Acc present", "n declined",
                            "Acc declined", "delta"], rows)))
    print("Association only. The corruption sweep "
          "(`experiments/analysis.py error-propagation`) is the interventional test.")
else:
    print(f"{PREDICTIONS} not found — skipping.")

## 6 · Manual assessment on a representative subset

`adjectives`/`neighbors` have no ground truth in ScanRefer, so this exports a CSV for hand labelling. The sample is stratified by target correctness (a uniform draw would be ~94% correct and show nothing about failure modes); `sampling_manifest.json` records the strata so rates re-weight back to population rates.

In [ ]:
import random, csv
rng = random.Random(SEED)
correct = [r for r in records if r["match"]["fuzzy"]]
wrong   = [r for r in records if not r["match"]["fuzzy"]]
n_wrong = min(int(round(SAMPLE_N * WRONG_FRACTION)), len(wrong))
n_right = min(SAMPLE_N - n_wrong, len(correct))
sample  = rng.sample(wrong, n_wrong) + rng.sample(correct, n_right)
rng.shuffle(sample)

manifest = {"split": SPLIT, "seed": SEED, "sample_size": len(sample),
            "wrong_fraction_requested": WRONG_FRACTION,
            "strata": {
                "target_wrong":   {"sampled": n_wrong, "population": len(wrong)},
                "target_correct": {"sampled": n_right, "population": len(correct)}}}
display(Markdown("**Sampling plan**\n\n" + table(
    ["stratum", "sampled", "population", "sampling rate", "population share"],
    [[k, v["sampled"], f"{v['population']:,}",
      f"{100*v['sampled']/v['population']:.2f}%",
      f"{100*v['population']/len(records):.2f}%"] for k, v in manifest["strata"].items()])))

COLS = ["scene_id","object_id","ann_id","object_name","description",
        "parsed_target","parsed_adjectives","parsed_neighbors",
        "target_auto_correct","stratum",
        "target_ok","adjectives_ok","neighbors_ok","error_code","notes"]
path = os.path.join(OUT_DIR, f"manual_assessment_{SPLIT}.csv")
with open(path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(COLS)
    for r in sample:
        w.writerow([r["scene_id"], r["object_id"], r["ann_id"], r["object_name"],
                    r["description"], r["raw"]["target"], r["raw"]["adjectives"],
                    r["raw"]["neighbors"], int(r["match"]["fuzzy"]),
                    "target_wrong" if not r["match"]["fuzzy"] else "target_correct",
                    "", "", "", "", ""])
json.dump(manifest, open(os.path.join(OUT_DIR, "sampling_manifest.json"), "w"), indent=2)
print(f"\nwrote {path}  ({len(sample)} rows)")
print(f"wrote {os.path.join(OUT_DIR, 'sampling_manifest.json')}")

In [ ]:
INSTRUCTIONS = """# Manual parse assessment — how to fill this in

Fill the five blank columns. Judge the parse against the DESCRIPTION, not against your
own idea of the object: the parser only ever sees the description.

  target_ok       1 = the parsed target is the object the description refers to
                  0 = it is not
  adjectives_ok   1 = the attributes are the referred object's, and are in the description
                  0 = wrong, invented, or belonging to another object
                  2 = "not mentioned" and the description really supplies no attributes
  neighbors_ok    same scale, for the spatial-relation phrases
  error_code      one of the codes below, blank when all three are fine
  notes           free text, anything the codes do not capture

Error codes
  WRONG_OBJECT     parsed the wrong object (often a landmark, not the target)
  MISSING          field left "not mentioned" although the description supplies it
  INVENTED         content that is not in the description
  PARTIAL          right idea, incomplete (dropped one of two attributes)
  MISATTRIBUTED    real content, attached to the wrong object
  VOCAB            correct in the description's own words, disagrees with object_name

A row where the description itself is ambiguous or wrong is still labelled against the
description; note it in `notes`.
"""
p = os.path.join(OUT_DIR, "instructions.md")
open(p, "w").write(INSTRUCTIONS)
print("wrote", p); print(); print(INSTRUCTIONS[:700])

### Re-weighting a filled sheet back to population rates

Run after the CSV is filled in — the sample over-samples failures, so raw rates aren't dataset rates.

In [ ]:
def reweight(filled_csv, manifest_path):
    """Sample rates and stratum-re-weighted population rates from a filled sheet."""
    man = json.load(open(manifest_path))
    rows = list(csv.DictReader(open(filled_csv, encoding="utf-8")))
    rows = [r for r in rows if any(r[f].strip() for f in
                                   ("target_ok", "adjectives_ok", "neighbors_ok"))]
    if not rows:
        print("No labelled rows yet — fill the CSV first."); return None
    pop_total = sum(s["population"] for s in man["strata"].values())
    out = {}
    for field in ("target_ok", "adjectives_ok", "neighbors_ok"):
        per, sample_hits, sample_n = {}, 0, 0
        for name, s in man["strata"].items():
            vals = [r[field].strip() for r in rows
                    if r["stratum"] == name and r[field].strip() in ("0", "1", "2")]
            # 2 = correctly declined, which counts as correct
            hits = sum(1 for v in vals if v in ("1", "2"))
            per[name] = (hits, len(vals))
            sample_hits += hits; sample_n += len(vals)
        pop = sum((s["population"] / pop_total) * (per[n][0] / per[n][1])
                  for n, s in man["strata"].items() if per[n][1])
        out[field] = {"sample": 100*sample_hits/sample_n if sample_n else None,
                      "sample_n": sample_n, "population": 100*pop, "per_stratum": per}
    display(Markdown("**Manual assessment**\n\n" + table(
        ["field", "sample rate", "n labelled", "population (re-weighted)"],
        [[f, f"{v['sample']:.1f}%", v["sample_n"], f"{v['population']:.1f}%"]
         for f, v in out.items()])))
    print("Quote the POPULATION column. The sample over-samples failures by design.")
    return out

filled = os.path.join(OUT_DIR, f"manual_assessment_{SPLIT}.csv")
_ = reweight(filled, os.path.join(OUT_DIR, "sampling_manifest.json"))

## 7 · Summary

In [ ]:
L = [
 "# Parsing quality and encoding — summary", "",
 f"Split `{SPLIT}`, {len(records):,} annotations, parser GPT-4o-mini "
 f"(`{PARSE_RAW}`).", "",
 "## Encoding", "",
 "| stage | representation |", "|---|---|",
 "| 1 | LLM emits JSON: three **string** fields |",
 "| 2 | tokenised to word lists, capped at " +
 ", ".join(f"{f} {c}" for f, c in FIELD_MAX_TOKENS.items()) + " |",
 "| 3 | `_transform_parsed` -> `(cap, 300)` GloVe rows; OOV -> `<unk>`; padding rows zero |",
 "| 4 | lengths = `len(tokens)`, unclipped, into `pack_padded_sequence` |", "",
 f"- An absent field is the literal string `\"not mentioned\"` -> `{NOT_MENTIONED_TOKENS}`. "
 "Never `null`, `\"\"` or `[]`, all of which would crash the language encoder.",
 f"- Declined rates: " + ", ".join(f"**{f} {100*nm[f]/len(records):.2f}%**" for f in FIELDS) + ".",
 f"- Schema check over the split: "
 f"{'no empty field, no field over cap, every value a string' if not viol and not badtype else str(dict(viol))}.",
 f"- Malformed LLM output in the shipped cache: **none**. "
 f"{100*sum(mismatch.values())/(len(records)*3):.2f}% of fields differ from this repo's "
 "tokenizer, and every difference is hyphen handling, not a malformed parse.",
]
if oov_rate is not None:
    L.append(f"- Out-of-vocabulary rate into `<unk>`: **{oov_rate:.2f}%** of cache tokens.")
L += ["", "## Quality", "",
 "| criterion | target accuracy |", "|---|---|"] +  [f"| {c} | **{100*kinds[c]/len(records):.2f}%** |" for c in ("exact","substring","fuzzy")] +  ["", "Failure taxonomy of the 5.9% that miss the fuzzy criterion:", ""] +  ["| outcome | share |", "|---|---|"] +  [f"| {b} | {100*bucket[b]/len(records):.2f}% |" for b in order if bucket[b]] +  ["",
  f"**Genuine parser errors are {100*bucket[order[3]]/len(records):.2f}% of the split.** "
  "Most misses are a vocabulary gap between the description writer and the annotator "
  "(*desk* for an annotated `table`), where the parser copied the description correctly.",
  "", f"Manual assessment: {SAMPLE_N} annotations stratified by target correctness "
  f"(`{os.path.join(OUT_DIR, f'manual_assessment_{SPLIT}.csv')}`), with strata recorded "
  "so sample rates re-weight to population rates."]
summary = "\n".join(L)
display(Markdown(summary))
open(os.path.join(OUT_DIR, "parse_quality_summary.md"), "w").write(summary + "\n")
json.dump({"split": SPLIT, "n": len(records),
           "target_accuracy": {c: 100*kinds[c]/len(records) for c in ("exact","substring","fuzzy")},
           "taxonomy": {b: 100*bucket[b]/len(records) for b in order},
           "not_mentioned_rate": {f: 100*nm[f]/len(records) for f in FIELDS},
           "schema_violations": {f"{k[0]}_{k[1]}": v for k, v in viol.items()},
           "tokenizer_mismatch_pct": 100*sum(mismatch.values())/(len(records)*3),
           "oov_pct": oov_rate},
          open(os.path.join(OUT_DIR, "parse_quality.json"), "w"), indent=2)
print("\nwrote", os.path.join(OUT_DIR, "parse_quality_summary.md"))
print("wrote", os.path.join(OUT_DIR, "parse_quality.json"))